In [2]:
from google.cloud import bigquery

client = bigquery.Client.from_service_account_json(
    "../credentials/service_account.json"
)

print("Conexión exitosa")

Conexión exitosa


In [13]:
from google.cloud import bigquery

# Borramos la línea que sobreescribía el cliente. 
# Jupyter ya recuerda la variable 'client' que creaste y autenticaste en la celda [2].

# Guardamos todo tu SQL en una variable de texto (string)
query = """
CREATE SCHEMA IF NOT EXISTS retail_dw;

CREATE TABLE IF NOT EXISTS retail_dw.clientes (
    id_cliente INT64,
    nombre STRING,
    pais STRING
);

CREATE TABLE IF NOT EXISTS retail_dw.productos (
    id_producto INT64,
    producto STRING,
    categoria STRING
);

CREATE TABLE IF NOT EXISTS retail_dw.ventas (
    id_venta INT64,
    fecha_venta DATE,
    id_cliente INT64,
    id_producto INT64,
    sucursal STRING,
    monto NUMERIC
);
"""

# Ejecutamos el query usando el 'client' de la celda [2]
job = client.query(query)
job.result()

print("¡Esquema y tablas creados con éxito en BigQuery!")

¡Esquema y tablas creados con éxito en BigQuery!


In [3]:
import pandas as pd
import random
import os
from faker import Faker
from datetime import datetime, timedelta

# Inicializamos Faker
fake = Faker('es_MX')

# Asegurar que el directorio de salida exista
output_dir = '../datasets'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"Iniciando la generación de datos en la carpeta: {output_dir}/")

# 1. Generar 5,000 Clientes
print("Generando clientes.csv...")
paises_operacion = ['Guatemala', 'El Salvador', 'Honduras', 'Costa Rica']
clientes_data = []

for i in range(1, 5001):
    clientes_data.append({
        'id_cliente': i,
        'nombre': fake.name(),
        'pais': random.choice(paises_operacion)
    })

df_clientes = pd.DataFrame(clientes_data)
df_clientes.to_csv(os.path.join(output_dir, 'clientes.csv'), index=False)

# 2. Generar 500 Productos
print("Generando productos.csv...")
categorias = ['Electrónica', 'Hogar', 'Ropa', 'Deportes', 'Alimentos']
productos_data = []

for i in range(1, 501):
    productos_data.append({
        'id_producto': i,
        'producto': f'Producto_SKU_{i}', 
        'categoria': random.choice(categorias)
    })

df_productos = pd.DataFrame(productos_data)
df_productos.to_csv(os.path.join(output_dir, 'productos.csv'), index=False)

# 3. Generar 100,000 Ventas
print("Generando ventas.csv...")
sucursales_dict = {
    'Guatemala': ['Sucursal_GUA_Centro', 'Sucursal_GUA_Norte'],
    'El Salvador': ['Sucursal_ESV_San_Salvador', 'Sucursal_ESV_Santa_Ana'],
    'Honduras': ['Sucursal_HON_Tegucigalpa'],
    'Costa Rica': ['Sucursal_CRI_San_Jose', 'Sucursal_CRI_Alajuela']
}

ventas_data = []
start_date = datetime(2023, 1, 1)
end_date = datetime(2026, 6, 17)

for i in range(1, 100001):
    cliente = random.choice(clientes_data)
    sucursal = random.choice(sucursales_dict[cliente['pais']])
    random_days = random.randint(0, (end_date - start_date).days)
    fecha_venta = start_date + timedelta(days=random_days)
    
    ventas_data.append({
        'id_venta': i,
        'fecha_venta': fecha_venta.strftime('%Y-%m-%d'),
        'id_cliente': cliente['id_cliente'],
        'id_producto': random.randint(1, 500),
        'sucursal': sucursal,
        'monto': round(random.uniform(50.0, 5000.0), 2)
    })

df_ventas = pd.DataFrame(ventas_data)
df_ventas.to_csv(os.path.join(output_dir, 'ventas.csv'), index=False)

print(f"¡Proceso finalizado! Los archivos están listos en: {output_dir}/")

Iniciando la generación de datos en la carpeta: ../datasets/
Generando clientes.csv...
Generando productos.csv...
Generando ventas.csv...
¡Proceso finalizado! Los archivos están listos en: ../datasets/


In [7]:
from google.cloud import bigquery
import os

# Configuración inicial
dataset_id = 'retail_dw'

# Lista de archivos a cargar y sus tablas destino
archivos_a_cargar = {
    'clientes.csv': 'clientes',
    'productos.csv': 'productos',
    'ventas.csv': 'ventas'
}

# Configuración del job de carga
# Configuración del job de carga
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,  # Saltamos el encabezado del CSV
    autodetect=False,     # <-- APAGAMOS el autodetect para que respete nuestro esquema
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE # <-- Reemplaza los datos si ya existen, evitando duplicados
)

for archivo, tabla in archivos_a_cargar.items():
    # El '..' le dice a Python: "Sube una carpeta, luego entra a 'datasets' y busca el archivo"
    ruta_archivo = os.path.join('..', 'datasets', archivo)
    table_ref = f"{client.project}.{dataset_id}.{tabla}"
    
    print(f"Cargando {archivo} a {table_ref}...")
    
    with open(ruta_archivo, "rb") as source_file:
        job = client.load_table_from_file(source_file, table_ref, job_config=job_config)
    
    job.result()  # Esperamos a que la carga finalice
    
    table = client.get_table(table_ref)
    print(f"Carga completada. Total de filas en {tabla}: {table.num_rows}")

Cargando clientes.csv a project-e5dd6fd9-f250-4d97-810.retail_dw.clientes...
Carga completada. Total de filas en clientes: 5000
Cargando productos.csv a project-e5dd6fd9-f250-4d97-810.retail_dw.productos...
Carga completada. Total de filas en productos: 500
Cargando ventas.csv a project-e5dd6fd9-f250-4d97-810.retail_dw.ventas...
Carga completada. Total de filas en ventas: 100000


In [8]:
# ---------------------------------------------------------
# 1. Top 10 Productos
# ---------------------------------------------------------
print("--- 1. Top 10 Productos más vendidos ---")
query_top_productos = """
SELECT
    p.producto,
    SUM(v.monto) as total_ventas
FROM retail_dw.ventas v
JOIN retail_dw.productos p
ON v.id_producto = p.id_producto
GROUP BY p.producto
ORDER BY total_ventas DESC
LIMIT 10;
"""
# Ejecutamos la consulta y la mostramos
df_top_productos = client.query(query_top_productos).to_dataframe()
display(df_top_productos)


# ---------------------------------------------------------
# 2. Ventas por País
# ---------------------------------------------------------
print("\n--- 2. Ventas totales por País ---")
query_ventas_pais = """
SELECT
    c.pais,
    SUM(v.monto) as total_ventas
FROM retail_dw.ventas v
JOIN retail_dw.clientes c
ON v.id_cliente = c.id_cliente
GROUP BY c.pais
ORDER BY total_ventas DESC;
"""
df_ventas_pais = client.query(query_ventas_pais).to_dataframe()
display(df_ventas_pais)


# ---------------------------------------------------------
# 3. Ventas Mensuales
# ---------------------------------------------------------
print("\n--- 3. Tendencia de Ventas Mensuales ---")
query_ventas_mensuales = """
SELECT
    EXTRACT(MONTH FROM fecha_venta) as mes,
    SUM(monto) as ventas
FROM retail_dw.ventas
GROUP BY mes
ORDER BY mes;
"""
df_ventas_mensuales = client.query(query_ventas_mensuales).to_dataframe()
display(df_ventas_mensuales)

--- 1. Top 10 Productos más vendidos ---


C:\Users\nicoe\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,producto,total_ventas
0,Producto_SKU_149,621267.060000000
1,Producto_SKU_29,618381.770000000
2,Producto_SKU_264,600935.040000000
3,Producto_SKU_233,599112.750000000
4,Producto_SKU_371,598125.790000000
5,Producto_SKU_298,592368.340000000
6,Producto_SKU_61,591268.320000000
7,Producto_SKU_454,584856.540000000
8,Producto_SKU_20,582764.180000000
9,Producto_SKU_249,582577.460000000



--- 2. Ventas totales por País ---


C:\Users\nicoe\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,pais,total_ventas
0,Costa Rica,64058439.890000000
1,Honduras,63709987.610000000
2,El Salvador,63261908.610000000
3,Guatemala,61169469.160000000



--- 3. Tendencia de Ventas Mensuales ---


C:\Users\nicoe\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,mes,ventas
0,1,24268280.560000000
1,2,22168662.190000000
2,3,24775517.480000000
3,4,24198428.550000000
4,5,25033818.120000000
5,6,21643931.640000000
6,7,18067983.950000000
7,8,18651818.890000000
8,9,17805185.270000000
9,10,18448011.100000000
